# 03 — Feature Engineering: RFM & Churn Modeling Features

Two things happen here: (1) build RFM (Recency/Frequency/Monetary) features for customer segmentation, and (2) build a churn-prediction dataset using a **time-based split** — features are computed only from orders before a cutoff date, and the label is whether the customer ordered again *after* that cutoff. This avoids the classic mistake of leaking future behavior into the features.

In [1]:
import pandas as pd
import numpy as np

orders = pd.read_csv('data/processed/orders_clean.csv', parse_dates=['order_date'])
customers = pd.read_csv('data/processed/customers_clean.csv', parse_dates=['signup_date'])
orders_valid = orders[(orders['is_valid_price']) & (~orders['is_guest_checkout'])]
print(f'{len(orders_valid):,} valid, attributable order lines')

57,098 valid, attributable order lines


## RFM features (as of the latest date in the dataset)

In [2]:
max_date = orders_valid['order_date'].max()
rfm = orders_valid.groupby('customer_id').agg(
    recency_days=('order_date', lambda x: (max_date - x.max()).days),
    frequency=('order_id', 'count'),
    monetary=('revenue', 'sum'),
).reset_index()
rfm['r_score'] = pd.qcut(rfm['recency_days'].rank(method='first'), 4, labels=[4,3,2,1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 4, labels=[1,2,3,4]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'].rank(method='first'), 4, labels=[1,2,3,4]).astype(int)

def segment(row):
    if row.r_score >= 3 and row.f_score >= 3 and row.m_score >= 3:
        return 'Champions'
    if row.r_score >= 3 and row.f_score >= 2:
        return 'Loyal'
    if row.r_score <= 2 and row.f_score >= 3:
        return 'At Risk (was frequent)'
    if row.r_score <= 2 and row.f_score <= 2:
        return 'Lost'
    return 'Developing'

rfm['segment'] = rfm.apply(segment, axis=1)
rfm['segment'].value_counts()

segment
Lost                      3276
Champions                 2721
At Risk (was frequent)    2333
Loyal                     1879
Developing                1009
Name: count, dtype: int64

In [3]:
rfm.to_csv('data/processed/rfm_segments.csv', index=False)
rfm.groupby('segment')['monetary'].mean().sort_values(ascending=False).round(2)

segment
Champions                 4714.85
At Risk (was frequent)    2617.31
Loyal                      895.24
Lost                       631.69
Developing                 448.06
Name: monetary, dtype: float64

## Churn features — time-based split (no leakage)

- `max_date`: latest order date in the cleaned dataset
- `cutoff = max_date - 180 days`: start of the "did they come back?" observation window
- `feature_cutoff = cutoff - 180 days`: features use **only** orders before this date
- eligible customers: signed up before `feature_cutoff`, so everyone gets a fair pre-cutoff window
- label: 1 if the customer placed **zero** orders in `(cutoff, max_date]`

In [4]:
cutoff = max_date - pd.Timedelta(days=180)
feature_cutoff = cutoff - pd.Timedelta(days=180)

eligible = customers[customers['signup_date'] <= feature_cutoff]
pre = orders_valid[(orders_valid['order_date'] <= cutoff) & (orders_valid['customer_id'].isin(eligible['customer_id']))]
post = orders_valid[(orders_valid['order_date'] > cutoff) & (orders_valid['order_date'] <= max_date)]

print(f'Eligible customers: {len(eligible):,}')
print(f'Cutoff: {cutoff.date()}   Feature cutoff: {feature_cutoff.date()}   Max date: {max_date.date()}')

Eligible customers: 7,756
Cutoff: 2025-07-04   Feature cutoff: 2025-01-05   Max date: 2025-12-31


In [5]:
feat = pre.groupby('customer_id').agg(
    frequency=('order_id', 'count'),
    monetary=('revenue', 'sum'),
    avg_order_value=('revenue', 'mean'),
    recency_days=('order_date', lambda x: (cutoff - x.max()).days),
    avg_discount=('discount_pct', 'mean'),
    n_categories=('product_id', 'nunique'),
    total_quantity=('quantity', 'sum'),
    first_order=('order_date', 'min'),
).reset_index()

window_mid = feature_cutoff + (cutoff - feature_cutoff) / 2
recent = pre[pre['order_date'] > window_mid].groupby('customer_id').size()
early = pre[pre['order_date'] <= window_mid].groupby('customer_id').size()
feat['recent_orders'] = feat['customer_id'].map(recent).fillna(0)
feat['early_orders'] = feat['customer_id'].map(early).fillna(0)
feat['engagement_trend'] = feat['recent_orders'] - feat['early_orders']
feat['tenure_days_at_cutoff'] = (cutoff - feat['first_order']).dt.days

feat = feat.merge(customers[['customer_id','segment','acquisition_channel']], on='customer_id', how='left')
churned = set(pre['customer_id']) - set(post['customer_id'])
feat['churned'] = feat['customer_id'].isin(churned).astype(int)

print(f"Churn rate in the 180-day observation window: {feat['churned'].mean()*100:.2f}%")
feat.to_csv('data/processed/churn_features.csv', index=False)
feat.head()

Churn rate in the 180-day observation window: 18.96%


,customer_id,frequency,monetary,avg_order_value,recency_days,avg_discount,n_categories,total_quantity,first_order,recent_orders,early_orders,engagement_trend,tenure_days_at_cutoff,segment,acquisition_channel,churned
0,4.0,1,74.80,74.800000,52,0.005000,1,2,2025-05-13,1.0,0.0,1.0,52,Consumer,Paid Search,1
1,6.0,4,1700.19,425.047500,42,0.039250,4,5,2023-12-16,1.0,3.0,-2.0,566,Consumer,Paid Search,0
2,7.0,2,69.96,34.980000,99,0.042000,2,4,2024-12-28,0.0,2.0,-2.0,188,Consumer,Paid Search,0
3,8.0,3,274.48,91.493333,23,0.033333,3,4,2025-03-04,2.0,1.0,1.0,122,Consumer,Social,1
4,10.0,2,242.66,121.330000,170,0.055000,2,2,2024-12-26,0.0,2.0,-2.0,190,Consumer,Email,0
